<a href="https://colab.research.google.com/github/MBilalQ/FinMMEval-Lab-2026/blob/main/FinnMMEval_Task3_Qwen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install transformers torch pandas datasets

In [26]:
import pandas as pd
import numpy as np
import torch
from transformers import pipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
import re

In [3]:
# 1. Initialize the FinBERT Pipeline
# Setting device=0 ensures it utilizes the free T4 GPU in Colab instead of the CPU
print("Loading FinBERT model...")
device = 0 if torch.cuda.is_available() else -1
finbert = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    device=device
)

Loading FinBERT model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
def get_sentiment_score(news_data):
    """
    Takes news data (handles both lists and strings), runs it through FinBERT,
    and returns a numerical score.
    """
    # Fix: If the dataset stores daily news as a list of articles, join them into one string
    if isinstance(news_data, (list, tuple)):
        # Join non-empty items with a space
        news_text = " ".join([str(item) for item in news_data if item])
    else:
        news_text = str(news_data)

    # Handle truly empty or missing text
    if not news_text or len(news_text.strip()) == 0 or news_text.strip().lower() in ['nan', 'none']:
        return 0.0

    try:
        # Pass to FinBERT (truncation handles texts longer than 512 tokens)
        result = finbert(news_text, padding=True, truncation=True, max_length=512)[0]
        label = result['label']

        if label == 'positive':
            return 1.0
        elif label == 'negative':
            return -1.0
        else:
            return 0.0

    except Exception as e:
        print(f"Error processing text: {e}")
        return 0.0

In [5]:
# 3. Load the BTC split specifically
print("Loading BTC dataset...")
dataset_btc = load_dataset("TheFinAI/CLEF_Task3_Trading", split="BTC")
df_btc = dataset_btc.to_pandas()

Loading BTC dataset...


In [6]:
# 4. Apply the Scoring Function
print("Calculating sentiment scores... ")
# df_sample = df_btc.head(20).copy()
df_btc['news_sentiment'] = df_btc['news'].apply(get_sentiment_score)

# View the results
print("\n--- BTC Sentiment Results ---")
print(df_btc[['date', 'news_sentiment']].head(10))

Calculating sentiment scores... 


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



--- BTC Sentiment Results ---
         date  news_sentiment
0  2025-08-01            -1.0
1  2025-08-02            -1.0
2  2025-08-03             0.0
3  2025-08-04             1.0
4  2025-08-05             1.0
5  2025-08-06             1.0
6  2025-08-07             0.0
7  2025-08-08             1.0
8  2025-08-09             1.0
9  2025-08-10             1.0


In [7]:
# 5. Load the TSLA split specifically
print("Loading TSLA dataset...")
dataset_tsla = load_dataset("TheFinAI/CLEF_Task3_Trading", split="TSLA")
df_tsla = dataset_tsla.to_pandas()

Loading TSLA dataset...


In [8]:
# 6. Apply the Scoring Function
print("Calculating sentiment scores... ")
# df_sample = df_btc.head(20).copy()
df_tsla['news_sentiment'] = df_tsla['news'].apply(get_sentiment_score)

# View the results
print("\n--- TSLA Sentiment Results ---")
print(df_tsla[['date', 'news_sentiment']].head(10))

Calculating sentiment scores... 

--- TSLA Sentiment Results ---
         date  news_sentiment
0  2025-08-01            -1.0
1  2025-08-02            -1.0
2  2025-08-03             0.0
3  2025-08-04             0.0
4  2025-08-05             0.0
5  2025-08-06             0.0
6  2025-08-07             0.0
7  2025-08-08             0.0
8  2025-08-09            -1.0
9  2025-08-10             1.0


Running The Model

In [16]:
!pip install -U bitsandbytes accelerate transformers

In [18]:
# 1. Quantization Setup (The "Magic" for Colab T4)
print("Configuring 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

Configuring 4-bit quantization...


In [19]:
# 2. Load the LLM and Tokenizer
model_id = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto" # Automatically maps to your T4 GPU
)

Loading Qwen/Qwen2.5-7B-Instruct...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [23]:
# 3. Define the Zero-Shot Prompt & Inference Function
def get_decision_and_reasoning(row):
    """
    Constructs the prompt Prompt_i = N_i + P_i + Q_i and extracts decision/reasoning.
    """
    # N_i: News and Sentiment (assuming you ran the previous FinBERT step)
    # If news_sentiment isn't calculated yet, default to 0 for this demo
    sentiment = row.get('news_sentiment', 0.0)

    # P_i: Price and Momentum
    price = row['prices']
    momentum = row['momentum']

    # Q_i: The Query
    prompt = f"""You are a quantitative financial agent. Analyze the following data for an asset:
- Current Price: {price}
- Momentum: {momentum}
- Daily News Sentiment Score: {sentiment} (Scale: -1 highly negative to 1 highly positive)

Based on this, make a trading decision for tomorrow.
Provide your response strictly in the following format:
Decision: [BUY, SELL, or HOLD]
Reasoning: [One brief sentence explaining why]
"""

    # Format for the chat model
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Generate output
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=50,
        temperature=0.1, # Low temperature for more deterministic/consistent output
        do_sample=True
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # 4. Parse the output using Regex
    decision_match = re.search(r'Decision:\s*(BUY|SELL|HOLD)', response, re.IGNORECASE)
    reasoning_match = re.search(r'Reasoning:\s*(.*)', response, re.IGNORECASE)

    decision_text = decision_match.group(1).upper() if decision_match else "NO DECISION"
    reasoning_text = reasoning_match.group(1).strip() if reasoning_match else "Could not parse reasoning."

    return pd.Series([decision_text, reasoning_text])

In [28]:
print("Running LLM Zero-Shot Predictions...")
df_test = df_btc.copy()

# Apply the prompt function
df_test[['raw_decision', 'reasoning']] = df_test.apply(get_decision_and_reasoning, axis=1)

# 5. Map Decisions to Numerical Values
decision_map = {'BUY': 1, 'SELL': -1, 'HOLD': 0.5, 'NO DECISION': 0}
df_test['action'] = df_test['raw_decision'].map(decision_map).fillna(0)

Running LLM Zero-Shot Predictions...


In [29]:
# 6. Calculate Financial Metrics
# First, calculate the daily return of the asset.
# If we make a decision today, our return relies on the price change of the *next* day.
df_test['daily_return_asset'] = df_test['prices'].pct_change().shift(-1)

# Calculate strategy returns: returns_i = daily_i * action_i
df_test['strategy_return'] = df_test['daily_return_asset'] * df_test['action']

# Drop the last row since its future return is NaN
df_valid = df_test.dropna(subset=['strategy_return']).copy()

# Calculate Cumulative Return (CR)
# Formula: Product of (1 + return) - 1
cumulative_return = (1 + df_valid['strategy_return']).prod() - 1

# Calculate Sharpe Ratio (SR)
# Formula: (Mean / Std) * sqrt(252 trading days)
mean_return = df_valid['strategy_return'].mean()
std_return = df_valid['strategy_return'].std()

if std_return > 0:
    sharpe_ratio = (mean_return / std_return) * np.sqrt(252)
else:
    sharpe_ratio = 0.0

print("\n--- Trading Decisions Output ---")
print(df_valid[['date', 'prices', 'raw_decision', 'action', 'reasoning', 'strategy_return']].head(10))

print("\n--- Performance Metrics ---")
print(f"Cumulative Return (CR): {cumulative_return:.4%}")
print(f"Sharpe Ratio (SR): {sharpe_ratio:.4f}")


--- Trading Decisions Output ---
         date     prices raw_decision  action  \
0  2025-08-01  113343.12         SELL    -1.0   
1  2025-08-02  112784.96         SELL    -1.0   
2  2025-08-03  114126.62         SELL    -1.0   
3  2025-08-04  115176.19          BUY     1.0   
4  2025-08-05  114191.49          BUY     1.0   
5  2025-08-06  115054.66          BUY     1.0   
6  2025-08-07  117499.31         HOLD     0.5   
7  2025-08-08  116929.14          BUY     1.0   
8  2025-08-09  116469.00          BUY     1.0   
9  2025-08-10  118980.47          BUY     1.0   

                                           reasoning  strategy_return  
0  The bearish momentum and negative daily news s...         0.004925  
1  The asset is in a bearish momentum and has a h...        -0.011896  
2  The asset is in a bearish momentum and has a n...        -0.009197  
3  The asset shows bullish momentum and positive ...        -0.008550  
4  The asset shows bullish momentum and positive ...         0.007

Tesla

In [30]:
print("Running LLM Zero-Shot Predictions...")
df_test_tsla = df_tsla.copy()

# Apply the prompt function
df_test_tsla[['raw_decision', 'reasoning']] = df_test_tsla.apply(get_decision_and_reasoning, axis=1)

# 5. Map Decisions to Numerical Values
decision_map = {'BUY': 1, 'SELL': -1, 'HOLD': 0.5, 'NO DECISION': 0}
df_test_tsla['action'] = df_test_tsla['raw_decision'].map(decision_map).fillna(0)

Running LLM Zero-Shot Predictions...


In [31]:
# 6. Calculate Financial Metrics
# First, calculate the daily return of the asset.
# If we make a decision today, our return relies on the price change of the *next* day.
df_test_tsla['daily_return_asset'] = df_test_tsla['prices'].pct_change().shift(-1)

# Calculate strategy returns: returns_i = daily_i * action_i
df_test_tsla['strategy_return'] = df_test_tsla['daily_return_asset'] * df_test_tsla['action']

# Drop the last row since its future return is NaN
df_valid = df_test_tsla.dropna(subset=['strategy_return']).copy()

# Calculate Cumulative Return (CR)
# Formula: Product of (1 + return) - 1
cumulative_return = (1 + df_valid['strategy_return']).prod() - 1

# Calculate Sharpe Ratio (SR)
# Formula: (Mean / Std) * sqrt(252 trading days)
mean_return = df_valid['strategy_return'].mean()
std_return = df_valid['strategy_return'].std()

if std_return > 0:
    sharpe_ratio = (mean_return / std_return) * np.sqrt(252)
else:
    sharpe_ratio = 0.0

print("\n--- Trading Decisions Output ---")
print(df_valid[['date', 'prices', 'raw_decision', 'action', 'reasoning', 'strategy_return']].head(10))

print("\n--- Performance Metrics ---")
print(f"Cumulative Return (CR): {cumulative_return:.4%}")
print(f"Sharpe Ratio (SR): {sharpe_ratio:.4f}")


--- Trading Decisions Output ---
         date      prices raw_decision  action  \
0  2025-08-01  302.630005         SELL    -1.0   
1  2025-08-02  302.630005         SELL    -1.0   
2  2025-08-03  302.630005         SELL    -1.0   
3  2025-08-04  309.260010         HOLD     0.5   
4  2025-08-05  308.720001         HOLD     0.5   
5  2025-08-06  319.910004         HOLD     0.5   
6  2025-08-07  322.269989         HOLD     0.5   
7  2025-08-08  329.649994         HOLD     0.5   
8  2025-08-09  329.649994         SELL    -1.0   
9  2025-08-10  329.649994          BUY     1.0   

                                           reasoning  strategy_return  
0  The asset shows bearish momentum and negative ...        -0.000000  
1  The asset shows bearish momentum and negative ...        -0.000000  
2  The asset is in a bearish momentum and has a n...        -0.021908  
3  The asset shows bullish momentum but has a neu...        -0.000873  
4  The asset shows bullish momentum but has a neu...   